# Agentic + RAG metrics: offline test suite (CPU runtime)

No GPU and no model needed. Runs every offline test in `tests/agentic_suite/`:
hand-computed edge cases for the 6 agentic and 8 RAG metrics, every tool-call wire format, and
`ak.evaluate()` end to end through `precomputed`, prompt mode, and the real `api:` / `agent:` backends
against local mock HTTP servers. Then regenerates the per-metric Markdown reports.

In [ ]:
from google.colab import files
up = files.upload()                       # a zip of the repo (with tests/agentic_suite)
!rm -rf /content/auditkit && mkdir -p /content/auditkit && unzip -q -o {next(iter(up))} -d /content/auditkit
%cd /content/auditkit
!pip -q install -e . pytest requests

## 1. Run the suite
Expect everything to pass; the live tests show as *skipped* (they need `AK_LIVE=1` and a model server).

In [ ]:
!python -m pytest -o addopts= tests/agentic_suite -p no:cacheprovider -q -rs --junitxml=/content/offline_junit.xml

## 2. Per-test listing (what each check is)

In [ ]:
!python -m pytest -o addopts= tests/agentic_suite -p no:cacheprovider -q --collect-only | head -400

## 3. Regenerate the per-metric reports

In [ ]:
!python -m tests.agentic_suite.make_metric_reports /content/offline_junit.xml
from IPython.display import Markdown, display
display(Markdown(open("tests/agentic_suite/metric_reports/README.md").read()))

In [ ]:
!cd /content/auditkit/tests/agentic_suite && zip -qr /content/metric_reports.zip metric_reports
files.download("/content/metric_reports.zip")

## 4. (Optional) the whole AuditKit test suite
Some older tests need torch/transformers extras and are skipped or fail without them.

In [ ]:
!python -m pytest -o addopts= tests -p no:cacheprovider -q -x --ignore=tests/agentic_suite 2>&1 | tail -20